Task A — Record Extraction and Search (CO1)
University administration records

In [1]:
data = {
"U01":"A separate matter is marked received in the log. Ref: UNI/26/501.",
"U02":"A new note concerning fee was added to the record. Ref: UNI/26/502.",
"U03":"The academic team opened a fresh inquiry about fee. Reference UNI/26/503.",
"U04":"A caller mentioned fee but supplied few details. Ref: UNI/26/504. Contact: person2004@example.invalid.",
"U05":"An inquiry about fee reached the desk. Ref: UNI/26/505.",
"U06":"The help desk forwarded the message to a colleague. Reference UNI/26/506. Logged: 02/09/2026.",
"U07":"Office personnel added a note to the logbook. Ref: uni/26/507.",
"U08":"The fee request is received in the student portal. Ref: UNI/26/508. Contact: person2008@example.invalid.",
"U09":"FEE RECEIVED! Please verify this item promptly. Reference UNI/26/509.",
"U10":"A student contacted the office regarding fee paperwork. Ref: UNI/26/510.",
"U11":"A label reads received; no topic was specified. Ref: UNI/26/511.",
"U12":"The student portal shows received against an unrelated entry. Reference UNI/26/512. Contact: person2012@example.invalid. Logged: 08/09/2026.",
"U13":"Staff recorded an unrelated inquiry during the shift. Ref: UNI/26/513.",
"U14":"This entry describes an ordinary internal update. Ref: uni/26/514.",
"U15":"The academic record for fee was marked received by the office. Reference UNI/26/515.",
"U16":"A caller left comments about another matter. Ref: UNI/26/516. Contact: person2016@example.invalid.",
"U17":"DO NOT ignore the fee issue: received was recorded. Ref: UNI/26/517.",
"U18":"A student asked the registrar about charges in a separate case. Reference UNI/26/518. Logged: 14/09/2026."
}

In [9]:
import nltk
import re
from nltk.corpus import stopwords
from collections import defaultdict

In [8]:
reference_pattern = r'\bUNI/\d{2}/\d{3}\b'

reference_index = {}

for record_id, message in data.items():
    match = re.search(reference_pattern, message, re.IGNORECASE)

    if match:
        reference_code = match.group().upper()
        reference_index[reference_code] = record_id

for reference, record_id in reference_index.items():
    print(record_id, ":", reference)


U01 : UNI/26/501
U02 : UNI/26/502
U03 : UNI/26/503
U04 : UNI/26/504
U05 : UNI/26/505
U06 : UNI/26/506
U07 : UNI/26/507
U08 : UNI/26/508
U09 : UNI/26/509
U10 : UNI/26/510
U11 : UNI/26/511
U12 : UNI/26/512
U13 : UNI/26/513
U14 : UNI/26/514
U15 : UNI/26/515
U16 : UNI/26/516
U17 : UNI/26/517
U18 : UNI/26/518


In [19]:
reference_index

{'UNI/26/501': 'U01',
 'UNI/26/502': 'U02',
 'UNI/26/503': 'U03',
 'UNI/26/504': 'U04',
 'UNI/26/505': 'U05',
 'UNI/26/506': 'U06',
 'UNI/26/507': 'U07',
 'UNI/26/508': 'U08',
 'UNI/26/509': 'U09',
 'UNI/26/510': 'U10',
 'UNI/26/511': 'U11',
 'UNI/26/512': 'U12',
 'UNI/26/513': 'U13',
 'UNI/26/514': 'U14',
 'UNI/26/515': 'U15',
 'UNI/26/516': 'U16',
 'UNI/26/517': 'U17',
 'UNI/26/518': 'U18'}

In [10]:
inverted_index = defaultdict(set)

for record_id, message in data.items():
    words = re.findall(r'\b[a-z0-9]+\b', message.lower())
    for word in set(words):
        inverted_index[word].add(record_id)

print("Number of unique words:", len(inverted_index))

print("\nRecords containing fee:")
print(sorted(inverted_index.get("fee", set())))

print("\nRecords containing received:")
print(sorted(inverted_index.get("received", set())))

Number of unique words: 115

Records containing fee:
['U02', 'U03', 'U04', 'U05', 'U08', 'U09', 'U10', 'U15', 'U17']

Records containing received:
['U01', 'U08', 'U09', 'U11', 'U12', 'U15', 'U17']


In [23]:
words

['a',
 'student',
 'asked',
 'the',
 'registrar',
 'about',
 'charges',
 'in',
 'a',
 'separate',
 'case',
 'reference',
 'uni',
 '26',
 '518',
 'logged',
 '14',
 '09',
 '2026']

In [24]:
inverted_index

defaultdict(set,
            {'501': {'U01'},
             'separate': {'U01', 'U18'},
             'ref': {'U01',
              'U02',
              'U04',
              'U05',
              'U07',
              'U08',
              'U10',
              'U11',
              'U13',
              'U14',
              'U16',
              'U17'},
             'received': {'U01', 'U08', 'U09', 'U11', 'U12', 'U15', 'U17'},
             'log': {'U01'},
             'matter': {'U01', 'U16'},
             'uni': {'U01',
              'U02',
              'U03',
              'U04',
              'U05',
              'U06',
              'U07',
              'U08',
              'U09',
              'U10',
              'U11',
              'U12',
              'U13',
              'U14',
              'U15',
              'U16',
              'U17',
              'U18'},
             'in': {'U01', 'U08', 'U18'},
             'marked': {'U01', 'U15'},
             'is': {'U01', 'U08'},
       

In [11]:
def search_records(query):

    pattern = r'^\s*(\w+)\s+AND\s+(NOT\s+)?(\w+)\s*$'
    match = re.match(pattern, query, re.IGNORECASE)

    if not match:
        print("Invalid query. Use: term AND term or term AND NOT term")
        return

    term1, not_operator, term2 = match.groups()

    term1 = term1.lower()
    term2 = term2.lower()

    records1 = inverted_index.get(term1, set())
    records2 = inverted_index.get(term2, set())

    if not_operator:
        matching_ids = records1 - records2
    else:

        matching_ids = records1 & records2

    for record_id, message in data.items():
        if record_id in matching_ids:
            print("\nRecord ID:", record_id)
            print("Original message:", message)

    if not matching_ids:
        print("No matching records found.")

    return matching_ids

In [27]:
search_records("fee AND received")


Record ID: U08
Original message: The fee request is received in the student portal. Ref: UNI/26/508. Contact: person2008@example.invalid.

Record ID: U09
Original message: FEE RECEIVED! Please verify this item promptly. Reference UNI/26/509.

Record ID: U15
Original message: The academic record for fee was marked received by the office. Reference UNI/26/515.

Record ID: U17
Original message: DO NOT ignore the fee issue: received was recorded. Ref: UNI/26/517.


{'U08', 'U09', 'U15', 'U17'}

In [13]:
search_records("fee AND NOT received")


Record ID: U02
Original message: A new note concerning fee was added to the record. Ref: UNI/26/502.

Record ID: U03
Original message: The academic team opened a fresh inquiry about fee. Reference UNI/26/503.

Record ID: U04
Original message: A caller mentioned fee but supplied few details. Ref: UNI/26/504. Contact: person2004@example.invalid.

Record ID: U05
Original message: An inquiry about fee reached the desk. Ref: UNI/26/505.

Record ID: U10
Original message: A student contacted the office regarding fee paperwork. Ref: UNI/26/510.


{'U02', 'U03', 'U04', 'U05', 'U10'}

In [14]:
def retrieve_by_reference(reference):

    reference = reference.strip().upper()
    record_id = reference_index.get(reference)

    if record_id is None:
        print("Reference code not found.")
        return

    print("Reference code:", reference)
    print("Record ID:", record_id)
    print("Original message:", data[record_id])


In [22]:
retrieve_by_reference("uni/26/503")

Reference code: UNI/26/503
Record ID: U03
Original message: The academic team opened a fresh inquiry about fee. Reference UNI/26/503.


In [29]:
search_records("fee AND worse")

No matching records found.


set()

U:18 "**A student asked the registrar about charges in a separate case. Reference UNI/26/518. Logged: 14/09/2026.**"

Suppose a user searches for fee. The search system checks whether the exact word fee exists in the message. However, U18 contains the word “charges” instead of “fee”.

Although charges may be related to university fees, the inverted index does not understand synonyms or the meaning of words. Therefore, U18 will not appear in the search results.

Task B — Statistical Language Modelling (CO2)

In [30]:
from collections import Counter, defaultdict
import math

In [31]:
texts = [
    "candidates submit form details",
    "staff share form details",
    "staff verify fee details",
    "candidates discuss results reports",
    "advisers verify form notices",
    "staff verify schedule updates",
    "candidates submit fee updates",
    "candidates submit schedule updates",
    "advisers share schedule details",
    "officers share records updates",
    "officers verify office notices",
    "candidates submit fee details",
    "candidates discuss office notices",
    "staff share records notices",
    "officers verify results reports",
    "candidates discuss records notices",
    "staff verify office notices",
    "advisers share fee updates"
]

print("Number of sentences:", len(texts))

Number of sentences: 18


In [32]:
tokenized_sentences = []

for sentence in texts:
    words = sentence.lower().split()

    words = ["<s>"] + words + ["</s>"]

    tokenized_sentences.append(words)

In [33]:
tokenized_sentences

[['<s>', 'candidates', 'submit', 'form', 'details', '</s>'],
 ['<s>', 'staff', 'share', 'form', 'details', '</s>'],
 ['<s>', 'staff', 'verify', 'fee', 'details', '</s>'],
 ['<s>', 'candidates', 'discuss', 'results', 'reports', '</s>'],
 ['<s>', 'advisers', 'verify', 'form', 'notices', '</s>'],
 ['<s>', 'staff', 'verify', 'schedule', 'updates', '</s>'],
 ['<s>', 'candidates', 'submit', 'fee', 'updates', '</s>'],
 ['<s>', 'candidates', 'submit', 'schedule', 'updates', '</s>'],
 ['<s>', 'advisers', 'share', 'schedule', 'details', '</s>'],
 ['<s>', 'officers', 'share', 'records', 'updates', '</s>'],
 ['<s>', 'officers', 'verify', 'office', 'notices', '</s>'],
 ['<s>', 'candidates', 'submit', 'fee', 'details', '</s>'],
 ['<s>', 'candidates', 'discuss', 'office', 'notices', '</s>'],
 ['<s>', 'staff', 'share', 'records', 'notices', '</s>'],
 ['<s>', 'officers', 'verify', 'results', 'reports', '</s>'],
 ['<s>', 'candidates', 'discuss', 'records', 'notices', '</s>'],
 ['<s>', 'staff', 'verify',

In [45]:
all_bigrams = []

for words in tokenized_sentences:
    for i in range(len(words) - 1):
        bigram = (words[i], words[i + 1])
        all_bigrams.append(bigram)

print("Total bigram occurrences:", len(all_bigrams))


Total bigram occurrences: 90


In [48]:
bigram_counts = Counter(all_bigrams)

print("Count of ('candidates', 'submit'):",
      bigram_counts[("candidates", "submit")])



Count of ('candidates', 'submit'): 4


In [58]:
bigram_counts

Counter({('<s>', 'candidates'): 7,
         ('candidates', 'submit'): 4,
         ('submit', 'form'): 1,
         ('form', 'details'): 2,
         ('details', '</s>'): 5,
         ('<s>', 'staff'): 5,
         ('staff', 'share'): 2,
         ('share', 'form'): 1,
         ('staff', 'verify'): 3,
         ('verify', 'fee'): 1,
         ('fee', 'details'): 2,
         ('candidates', 'discuss'): 3,
         ('discuss', 'results'): 1,
         ('results', 'reports'): 2,
         ('reports', '</s>'): 2,
         ('<s>', 'advisers'): 3,
         ('advisers', 'verify'): 1,
         ('verify', 'form'): 1,
         ('form', 'notices'): 1,
         ('notices', '</s>'): 6,
         ('verify', 'schedule'): 1,
         ('schedule', 'updates'): 2,
         ('updates', '</s>'): 5,
         ('submit', 'fee'): 2,
         ('fee', 'updates'): 2,
         ('submit', 'schedule'): 1,
         ('advisers', 'share'): 2,
         ('share', 'schedule'): 1,
         ('schedule', 'details'): 1,
         ('<s>', 

In [49]:
context_counts = Counter()

for previous_word, next_word in all_bigrams:
    context_counts[previous_word] += 1

print("Count of candidates:", context_counts["candidates"])
print("Count of submit:", context_counts["submit"])
print("Count of fee:", context_counts["fee"])

Count of candidates: 7
Count of submit: 4
Count of fee: 4


In [57]:
context_counts

Counter({'<s>': 18,
         'candidates': 7,
         'submit': 4,
         'form': 3,
         'details': 5,
         'staff': 5,
         'share': 5,
         'verify': 6,
         'fee': 4,
         'discuss': 3,
         'results': 2,
         'reports': 2,
         'advisers': 3,
         'notices': 6,
         'schedule': 3,
         'updates': 5,
         'officers': 3,
         'records': 3,
         'office': 3})

In [50]:
bigram_probabilities = {}

for (previous_word, next_word), count in bigram_counts.items():
    probability = count / context_counts[previous_word]

    bigram_probabilities[(previous_word, next_word)] = probability

print("P(submit | candidates) =",
      bigram_probabilities[("candidates", "submit")])

print("P(discuss | candidates) =",
      bigram_probabilities[("candidates", "discuss")])

print("P(fee | submit) =",
      bigram_probabilities[("submit", "fee")])

P(submit | candidates) = 0.5714285714285714
P(discuss | candidates) = 0.42857142857142855
P(fee | submit) = 0.5


In [52]:
bigram_probabilities

{('<s>', 'candidates'): 0.3888888888888889,
 ('candidates', 'submit'): 0.5714285714285714,
 ('submit', 'form'): 0.25,
 ('form', 'details'): 0.6666666666666666,
 ('details', '</s>'): 1.0,
 ('<s>', 'staff'): 0.2777777777777778,
 ('staff', 'share'): 0.4,
 ('share', 'form'): 0.2,
 ('staff', 'verify'): 0.6,
 ('verify', 'fee'): 0.16666666666666666,
 ('fee', 'details'): 0.5,
 ('candidates', 'discuss'): 0.42857142857142855,
 ('discuss', 'results'): 0.3333333333333333,
 ('results', 'reports'): 1.0,
 ('reports', '</s>'): 1.0,
 ('<s>', 'advisers'): 0.16666666666666666,
 ('advisers', 'verify'): 0.3333333333333333,
 ('verify', 'form'): 0.16666666666666666,
 ('form', 'notices'): 0.3333333333333333,
 ('notices', '</s>'): 1.0,
 ('verify', 'schedule'): 0.16666666666666666,
 ('schedule', 'updates'): 0.6666666666666666,
 ('updates', '</s>'): 1.0,
 ('submit', 'fee'): 0.5,
 ('fee', 'updates'): 0.5,
 ('submit', 'schedule'): 0.25,
 ('advisers', 'share'): 0.6666666666666666,
 ('share', 'schedule'): 0.2,
 ('sc

In [51]:
previous_word = "candidates"

next_word_probabilities = {}

for (word1, word2), probability in bigram_probabilities.items():
    if word1 == previous_word:
        next_word_probabilities[word2] = probability

for word, probability in sorted(
    next_word_probabilities.items(),
    key=lambda item: item[1],
    reverse=True
):
    print(f"{word}: {probability:.4f}")


submit: 0.5714
discuss: 0.4286


In [53]:
def sentence_probability(sentence):
    words = sentence.lower().split()
    words = ["<s>"] + words + ["</s>"]

    probability = 1.0

    for i in range(len(words) - 1):
        bigram = (words[i], words[i + 1])

        count = bigram_counts[bigram]
        denominator = context_counts[words[i]]

        if count == 0:
            return 0.0

        probability *= count / denominator

    return probability


sentence = "candidates submit fee updates"

prob = sentence_probability(sentence)

print("Sentence:", sentence)
print("Probability:", prob)

Sentence: candidates submit fee updates
Probability: 0.05555555555555555


In [54]:
def sentence_perplexity(sentence):
    words = sentence.lower().split()
    words = ["<s>"] + words + ["</s>"]

    log_probability_sum = 0.0
    number_of_transitions = len(words) - 1

    for i in range(len(words) - 1):
        previous_word = words[i]
        next_word = words[i + 1]

        count = bigram_counts[(previous_word, next_word)]
        denominator = context_counts[previous_word]

        if count == 0:
            return float("inf")

        probability = count / denominator
        log_probability_sum += math.log(probability)

    perplexity = math.exp(
        -log_probability_sum / number_of_transitions
    )

    return perplexity

In [55]:
sentence_a = "candidates submit schedule details"
sentence_b = "staff submit fee updates"

prob_a = sentence_probability(sentence_a)
prob_b = sentence_probability(sentence_b)

pp_a = sentence_perplexity(sentence_a)
pp_b = sentence_perplexity(sentence_b)

print("Sentence A:", sentence_a)
print("Probability:", prob_a)
print("Perplexity:", round(pp_a, 4))

print("\nSentence B:", sentence_b)
print("Probability:", prob_b)
print("Perplexity:", pp_b)

Sentence A: candidates submit schedule details
Probability: 0.018518518518518517
Perplexity: 2.2206

Sentence B: staff submit fee updates
Probability: 0.0
Perplexity: inf


In [56]:
print(bigram_counts[("staff", "submit")])

0


Since "**Staff submit**" bigram probability is Zero. Here the Probability of the sentence will be Zero. So while calculating the Perplexity of the it would be 1/0 so leads to Infinity.

and also since it as unsmoothend model there are chances to occur zero probability in bigram model.

If the probability of the given word or sentence is higher then perplexity of the model is low, this explains that the model predicts the next words with relatively high confidence.